# HIT-UAV × YOLO26 红外无人机弱小目标检测 — Colab Pro 总控启动台

**使用前 2 步检查（避免白烧 Compute Units）**：
1. 右上角「更改运行时类型」→ 选 **L4 GPU**（约 1.94 CU/h，22.5GB 显存 + 53GB 内存）。
2. 左侧钥匙图标 **Secrets（密钥）** → 添加密钥 `GITHUB_TOKEN`（勾选允许此笔记本访问）。

**推荐执行顺序**：
- **Phase 0 烟雾测试（约 2 分钟 / <0.06 CU）**：`RUN_TRAINING = True`、`SMOKE_TEST = True`、`EXP_LIST = ["p0_yolo26n_musgd"]`，确认训练 + 评测 + Drive 归档全链路无报错。
- **Phase 0 正式基线摸底（约 1.5 小时 / ~3 CU）**：`SMOKE_TEST = False`、`EXP_LIST = ["p0_yolo26n_musgd", "p0_yolo26n_sgd", "p0_yolo11n", "p0_yolov8n"]`、`AUTO_UNASSIGN = True`。
- **Phase 1 改动消融**：切换 `EXP_LIST` 为 `["p1_mod_a", "p1_mod_b", "p1_mod_c1", "p1_mod_c12", "p1_full_abc1", "p1_full_abc12"]`。

In [ ]:
# =============================================================================
# 🎛️ 总控开关区（平时改实验只需要改这里！）
# =============================================================================

# 1. 你的 GitHub 仓库
GITHUB_REPO = "hxttty/ir-yolo26-paper"
GITHUB_BRANCH = "main"
USE_PRIVATE_TOKEN = True   # 私有仓库设为 True（从 Colab Secrets 读取 GITHUB_TOKEN）

# 2. Google Drive 持久存储根目录（权重、日志、数据集缓存、结果总表都在这里）
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/ir-yolo26-paper"
RAW_HITUAV_PATH = None

# 3. 任务总开关
RUN_PHASE0_INSPECT = False   # 已做完可保持 False
RUN_PREPARE_DATA   = True    # Drive 已有缓存包 hit_uav_bundle.zip，每次开新机仅需 3 秒秒恢复
RUN_TRAINING       = True    # 开启训练队列

# 4. 实验队列与安全开关（支持一次排队跑多个预设！）
# --- 推荐 V3 执行顺序 ---
#   Step A (1 分钟烟雾验收): SMOKE_TEST=True, AUTO_UNASSIGN=False, EXP_LIST=["v3_e7_full", "v3_e7b_full_sd"]
#   Step B (V3 Batch 1 核心 Pillar 1 门控 ~2h / ~4.5 CU): SMOKE_TEST=False, AUTO_UNASSIGN=True,
#       EXP_LIST=["v3_e0_base", "v3_e1_p1_distill", "v3_e2_p1_piou", "v3_e2b_p1_sdloss"]
#   Step C (V3 Batch 2 结构正交+全量门控 ~2.5h / ~5.5 CU): 视 Batch 1 胜者择优跑 e3 + e4 + e5 + e6 + e7
EXP_LIST = [
    "auto_phase3_overnight",
]
SMOKE_TEST    = False  # auto_phase3_overnight 自带强制 Gate 1 两轮冒烟熔断，正式过夜请保持 False！
SEED          = 42     # 5 种子验证时依次改 42 / 0 / 3407 / 2025 / 2026
RESUME        = True   # True=若上次中途断连，自动从 Drive 的 last.pt + best.pt + kd_state.pt 断点续训
EXPORT_ONNX   = False  # True=训完顺便导出 ONNX 并测 CPU 端单图延迟（正式出表时打开）
AUTO_UNASSIGN = True   # 跑完或报错后自动销毁运行时停表省 CU！


In [ ]:
# =============================================================================
# Cell 2: 挂载 Google Drive + 检查 GPU 硬件 + 实时日志工具函数
# =============================================================================
import os
import subprocess
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
Path(DRIVE_PROJECT_DIR).mkdir(parents=True, exist_ok=True)
!nvidia-smi

def run_stream(cmd, cwd="/content/ir-yolo26-paper"):
    """Run command and stream stdout/stderr LIVE to Colab cell output."""
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    proc = subprocess.Popen(
        cmd, cwd=str(cwd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
    ret = proc.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, cmd)


In [ ]:
# =============================================================================
# Cell 3: 从 GitHub 拉取/更新最新代码（Token 仅走内存 Header，不落盘）+ 安装依赖
# =============================================================================
import base64
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/ir-yolo26-paper")

clean_url = f"https://github.com/{GITHUB_REPO}.git"
git_auth_args = []
if USE_PRIVATE_TOKEN:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if not token:
        raise ValueError("未读取到 Colab Secret `GITHUB_TOKEN`，请在左侧钥匙图标里添加并授权。")
    auth_b64 = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_auth_args = ["-c", f"http.extraheader=AUTHORIZATION: basic {auth_b64}"]

try:
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), *git_auth_args, "fetch", "origin", GITHUB_BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{GITHUB_BRANCH}"], check=True)
    else:
        subprocess.run(["git", *git_auth_args, "clone", "-b", GITHUB_BRANCH, clean_url, str(REPO_DIR)], check=True)
except subprocess.CalledProcessError as exc:
    raise RuntimeError(
        f"Git 拉取失败 (exit={exc.returncode})。请检查 GITHUB_REPO 拼写及 GITHUB_TOKEN 权限。"
    ) from None

%cd /content/ir-yolo26-paper
!git log -1 --oneline
!pip install -q -r requirements.txt


In [ ]:
# =============================================================================
# Cell 4: [可选] 实装 Ultralytics YOLO26 源码体检报告
# =============================================================================
from pathlib import Path

if RUN_PHASE0_INSPECT:
    audit_out = Path(DRIVE_PROJECT_DIR) / "audit" / "phase0_yolo26_source_audit.md"
    run_stream(["python", "-m", "src.inspect_yolo26", "--out-md", str(audit_out)])
    print(f"\n✅ 体检报告已保存到 Drive: {audit_out}")
else:
    print("[SKIP] RUN_PHASE0_INSPECT=False")


In [ ]:
# =============================================================================
# Cell 5: 准备 / 3 秒解压 HIT-UAV 数据集到 Colab 本地高速盘 (/content/datasets)
# =============================================================================
from pathlib import Path

DATASET_ROOT = Path("/content/datasets")
DRIVE_BUNDLE = Path(DRIVE_PROJECT_DIR) / "datasets" / "hit_uav_bundle.zip"
VIS_DIR      = Path(DRIVE_PROJECT_DIR) / "audit" / "sanity_vis"

if RUN_PREPARE_DATA:
    cmd = [
        "python", "-m", "src.prepare_hituav",
        "--out-root", str(DATASET_ROOT),
        "--drive-bundle", str(DRIVE_BUNDLE),
        "--vis-dir", str(VIS_DIR),
    ]
    if RAW_HITUAV_PATH:
        cmd += ["--raw-dir", str(RAW_HITUAV_PATH)]
    run_stream(cmd)
else:
    print("[SKIP] RUN_PREPARE_DATA=False")


In [ ]:
# =============================================================================
# Cell 6: 执行实验队列 + 实时日志 + Drive 存档 + try...finally 自动断连停表防烧 CU
# =============================================================================
import traceback
from google.colab import runtime

if RUN_TRAINING:
    failed_exps = []
    manual_interrupt = False
    try:
        for exp_name in EXP_LIST:
            print("\n" + "#" * 72)
            print(f"🚀 开始运行实验: {exp_name} (smoke_test={SMOKE_TEST}, seed={SEED})")
            print("#" * 72)
            cmd = [
                "python", "-m", "src.train_runner",
                "--exp", exp_name,
                "--dataset-root", "/content/datasets",
                "--work-dir", "/content/work",
                "--drive-dir", DRIVE_PROJECT_DIR,
                "--seed", str(SEED),
            ]
            if SMOKE_TEST:
                cmd.append("--smoke-test")
            if not RESUME:
                cmd.append("--no-resume")
            if EXPORT_ONNX:
                cmd.append("--export-onnx")
            try:
                run_stream(cmd)
            except KeyboardInterrupt:
                raise
            except Exception as exp_err:
                print(f"\n⚠️ 实验 {exp_name} 失败 ({exp_err})，日志已同步至 Drive，自动继续队列下一组！")
                failed_exps.append((exp_name, str(exp_err)))
        if failed_exps:
            raise RuntimeError(f"队列结束，以下实验出错：{failed_exps}")
        print("\n🎉 队列内所有实验全部顺利运行并归档完毕！")
    except KeyboardInterrupt:
        manual_interrupt = True
        print("\n⏸️ 检测到手动中断，保留当前 GPU 会话不自动销毁。")
        raise
    finally:
        if AUTO_UNASSIGN and not manual_interrupt:
            print("\n🛑 AUTO_UNASSIGN=True -> 正在自动释放 Colab GPU 运行时，停止扣费！")
            runtime.unassign()
else:
    print("[SKIP] RUN_TRAINING=False")
